# Gundam Isaac Lab Simulation

This notebook sets up Isaac Lab, loads the Gundam URDF , plays back the 'walk-forward' sample motion provided in the repository, and saves the output video to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "isaacsim[all,extscache]==6.0.0" --extra-index-url https://pypi.nvidia.com
!pip install pandas

In [ ]:
!rm -rf /content/gundam_robot
!git clone https://github.com/gundam-global-challenge/gundam_robot.git /content/gundam_robot
!sed -i 's/damping="3e2" friction="1e3"/damping="0.0" friction="0.0"/g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf
# Replace package:// path with absolute path so Isaac Sim can find the meshes
!sed -i 's|package://gundam_rx78_description|/content/gundam_robot/gundam_rx78_description|g' /content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf

In [ ]:
import torch
import numpy as np
import pandas as pd
import imageio
import os

# Load the sample CSV motion
csv_path = "/content/gundam_robot/gundam_rx78_control/sample/csv/walk-forward.csv"
df = pd.read_csv(csv_path)
# Clean up column names (remove leading spaces)
df.columns = df.columns.str.strip()

# Start Isaac Sim headlessly
from isaacsim import SimulationApp
simulation_app = SimulationApp({"headless": True})

from omni.isaac.core import World
from omni.isaac.urdf import _urdf
from omni.isaac.core.articulations import Articulation
import omni.isaac.core.utils.prims as prim_utils
from omni.isaac.sensor import Camera
import omni.kit.commands

# Initialize World (assuming CSV is recorded at 200Hz, we run sim at 200Hz)
sim_dt = 1.0 / 200.0
world = World(physics_dt=sim_dt, rendering_dt=1.0 / 60.0)
world.scene.add_default_ground_plane()

# Import URDF
urdf_interface = _urdf.acquire_urdf_interface()
import_config = _urdf.ImportConfig()
import_config.merge_fixed_joints = False
import_config.convex_decomp = False
import_config.import_inertia_tensor = True
import_config.fix_base = True
import_config.make_default_prim = False

urdf_path = "/content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf"
dest_path = "/World/Gundam"

print("Importing URDF This may take a moment.")
omni.kit.commands.execute(
    "URDFParseAndImportFile",
    urdf_path=urdf_path,
    import_config=import_config,
    dest_path=dest_path,
)

# Setup Articulation for sample motion
robot = Articulation(prim_path=dest_path, name="gundam")
world.scene.add(robot)

# Setup Camera
camera = Camera(prim_path="/World/Camera", position=np.array([15.0, -10.0, 10.0]), resolution=(640, 480))
camera.initialize()
camera.look_at(np.array([0.0, 0.0, 5.0]))
world.scene.add(camera)

world.reset()

# Map CSV columns to Isaac Sim joint indices
joint_names_isaac = robot.dof_names
csv_to_isaac_idx = {}
for csv_joint in df.columns:
    if csv_joint != 'time' and csv_joint in joint_names_isaac:
        csv_to_isaac_idx[csv_joint] = robot.get_dof_index(csv_joint)

# Simulation Loop
frames = []
render_accum = 0.0
render_dt = 1.0 / 60.0

print("Starting simulation playback...")
for idx, row in df.iterrows():
    # Apply target positions from CSV
    target_positions = robot.get_joint_positions()
    for csv_joint, isaac_idx in csv_to_isaac_idx.items():
        target_positions[isaac_idx] = row[csv_joint]
    
    robot.set_joint_position_targets(target_positions)
        
    world.step(render=False)
    
    # Render at 60 FPS instead of 200 FPS to save memory/time
    render_accum += sim_dt
    if render_accum >= render_dt:
        world.render()
        rgba = camera.get_rgba()
        if rgba is not None:
            frames.append(rgba[:, :, :3]) # Keep RGB, drop Alpha
        render_accum = 0.0

simulation_app.close()

# Save Video to Google Drive
video_path = "/content/drive/MyDrive/gundam_isaac_lab_walk.mp4"
if frames:
    imageio.mimwrite(video_path, frames, fps=60)
    print(f"Video saved to {video_path}")
else:
    print("No frames were captured.")
